# Heart Disease Risk Scoring: Clinical Interpretability vs. Accuracy
### End-to-End Portfolio Project for Cardiovascular Clinical Decision Support

> **Clinical Objective**: A cardiology referral triage clinic must evaluate patients with suspected Coronary Artery Disease (CAD >= 50% luminal stenosis). A false negative (missing a diseased patient) carries catastrophic clinical liability (unheralded myocardial infarction or sudden cardiac death). Conversely, a false positive simply results in a confirmatory non-invasive stress test or outpatient cardiology consultation.
>
> **Core Question**: Can a transparent, fully auditable linear model (Logistic Regression) match a complex black-box ensemble (XGBoost) closely enough in accuracy and calibration to be trusted in clinical practice?

**Dataset**: UCI Heart Disease (Cleveland Clinic Foundation, `id=45`, $n=303$, 13 features).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Ensure src modules can be imported
sys.path.insert(0, str(Path.cwd().parent))
from src.data import load_raw_data, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET_COL
from src.features import get_preprocessor, get_feature_names
from src.train import run_repeated_cv, build_logistic_regression_pipeline, build_xgboost_pipeline

X, y = load_raw_data()
df = pd.concat([X, y], axis=1)
print(f"Cohort size: {df.shape[0]} patients, {df.shape[1]-1} features")
print(f"CAD prevalence: {y.mean():.1%}")
df.head()

## 1. Clinical Features and Physiological Rationale

1. **`age`** (years): Cumulative exposure to arterial shear stress and lipid deposition.
2. **`sex`** (1 = Male, 0 = Female): Strong biological bias; estrogen provides vascular protective effects pre-menopause, causing males to present ~10 years earlier.
3. **`cp`** (Chest Pain Type: 1=Typical Angina, 2=Atypical Angina, 3=Non-anginal, 4=Asymptomatic): Classic exertional angina vs atypical symptoms.
4. **`trestbps`** (Resting Blood Pressure, mm Hg): Chronic hypertension induces endothelial microvascular injury.
5. **`chol`** (Serum Cholesterol, mg/dl): Atherogenic LDL circulating particles promote lipid core formation.
6. **`fbs`** (Fasting Blood Sugar > 120 mg/dl): Surrogate for diabetes mellitus, a coronary disease risk equivalent.
7. **`restecg`** (Resting ECG): 0=Normal, 1=ST-T abnormality, 2=Left Ventricular Hypertrophy (Estes criteria).
8. **`thalach`** (Peak Exercise Heart Rate, bpm): **Chronotropic incompetence** (inability to reach peak predicted heart rate) strongly signals ischemia.
9. **`exang`** (Exercise-Induced Angina): Direct symptomatic expression of coronary blood flow shortfall.
10. **`oldpeak`** (ST Depression, mm): Direct electrocardiographic marker of subendocardial ischemia.
11. **`slope`** (Peak ST Slope: 1=Upsloping, 2=Flat, 3=Downsloping): Horizontal/downsloping ST segments denote severe CAD.
12. **`ca`** (Major Vessels Colored by Fluoroscopy, 0-3): Direct anatomical count of calcified/obstructed coronaries.
13. **`thal`** (Thallium Perfusion Scintigraphy): 3=Normal, 6=Fixed Defect (prior infarct/scar), 7=Reversible Defect (ischemia at risk).

In [ ]:
# Statistical validation: Continuous features (Mann-Whitney U Test)
print("=== Continuous Risk Factors (Mann-Whitney U Tests) ===")
for feat in NUMERICAL_FEATURES:
    h = df[df[TARGET_COL] == 0][feat].dropna()
    d = df[df[TARGET_COL] == 1][feat].dropna()
    _, p = stats.mannwhitneyu(h, d)
    print(f"{feat:10s} | Healthy: {h.median():.1f} vs CAD: {d.median():.1f} | p-value: {p:.2e}")

## 2. Leak-Free Preprocessing Pipeline

To eliminate **data leakage**, imputation, scaling, and one-hot encoding must be fit **strictly on training folds** during cross-validation:
- Continuous: `SimpleImputer(strategy='median')` -> `StandardScaler()`
- Categorical: `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(drop='first', handle_unknown='ignore')`

In [ ]:
preprocessor = get_preprocessor()
X_trans = preprocessor.fit_transform(X)
features = get_feature_names(preprocessor)
print(f"Transformed shape: {X_trans.shape} (from {X.shape[1]} raw to {len(features)} encoded features)")
print("Features:", features)

## 3. Repeated Stratified Cross-Validation (5x5 = 25 Folds)

Small clinical cohorts ($n=303$) are subject to substantial validation variance. A single 80/20 train/test split can be misleading. We perform **5-fold CV repeated 5 times (25 independent folds)** with random seed 42 to establish robust uncertainty bounds (Mean ± Std).

In [ ]:
cv_summary_path = Path.cwd().parent / "reports" / "cv_metrics_summary.csv"
if cv_summary_path.exists():
    cv_df = pd.read_csv(cv_summary_path)
    display(cv_df)
else:
    lr_cv_df, _ = run_repeated_cv(build_logistic_regression_pipeline, X, y, "Logistic Regression")
    xgb_cv_df, _ = run_repeated_cv(build_xgboost_pipeline, X, y, "XGBoost")
    all_cv = pd.concat([lr_cv_df, xgb_cv_df], axis=0)
    print(all_cv.groupby('model').mean().round(3))

## 4. Clinical Threshold Tuning (Recall >= 90%)

At standard $p=0.50$ cutoff, models miss between 15% and 25% of patients with coronary artery disease. In clinical referral triage, **False Negatives** risk catastrophic patient outcomes. We sweep decision thresholds to calibrate the triage operating point to **>= 90% Recall**.

In [ ]:
import json
config_path = Path.cwd().parent / "models" / "threshold_config.json"
with open(config_path) as f:
    thresh_config = json.load(f)

print("Logistic Regression (Holdout Test n=61):")
print("  Default 0.50  :", thresh_config["models"]["logistic_regression"]["default_0.50"])
print("  Clinical Target:", thresh_config["models"]["logistic_regression"]["clinical_target"])

print("\nXGBoost (Holdout Test n=61):")
print("  Default 0.50  :", thresh_config["models"]["xgboost"]["default_0.50"])
print("  Clinical Target:", thresh_config["models"]["xgboost"]["clinical_target"])

## 5. Explainability: Transparent Odds Ratios vs Black-Box SHAP

- **Logistic Regression**: Interpretable Odds Ratios (OR = $\exp(\beta)$). Clinicians directly see multiplicative increases in CAD odds for every risk factor.
- **XGBoost**: TreeExplainer computes Shapley Additive Explanations (SHAP), providing global feature ranking and local waterfall breakdowns.

In [ ]:
odds_path = Path.cwd().parent / "reports" / "lr_odds_ratios.csv"
df_odds = pd.read_csv(odds_path)
print("=== Top Logistic Regression Risk Factors ===")
display(df_odds.tail(6))